# モデル方程式の数値

`poster/poster.md` の式を、半画面を一色で塗った 9 通りについて計算する。色の画素は 1 のまま、距離 $d$ は画像の横幅を 1 とした長さである。

ポスターが記号のままにしている数は、ここでは次のように置いた。$\alpha = r_{\max}/r_{\mathrm{ref}} = 100/50 = 2$。拡大は $k_s = 1$、$m = 8$ rad/s、$w = 5$ rad/s で、$\dot\theta = 0.4$ rad/s とする。膜は $\tau = 5$ ms、不応期 5 ms、$\Delta t = 1$ ms、$\tau_r = 50$ ms。1 秒積分し、発火率は最後の 100 ms の平均である。車輪の $k$ は未定なので、下の車輪は発火率の差そのもの（$k = 1$、単位は Hz）である。$\sigma_d = 0.15$。


In [1]:
from pathlib import Path
import sys

import cv2
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "flycar").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from flycar.loom import cells

SIGMA_D = 0.15
WIDTH, HEIGHT = 320, 180
ALPHA = 100.0 / 50.0
K_S, M_S, W_S = 1.0, 8.0, 5.0
THETA_DOT = 0.4
DT = 0.001
TAU = 0.005
TAU_R = 0.050
REFRAC = 0.005
RUN = 1.0


def expansion(theta_dot: float) -> float:
    return K_S / (1.0 + np.exp(-(1.0 / W_S) * (theta_dot - M_S)))


def spread(mask: np.ndarray) -> np.ndarray:
    color = np.asarray(mask, dtype=bool)
    if not np.any(color):
        return np.zeros(color.shape, dtype=np.float64)
    src = np.where(color, 0, 255).astype(np.uint8)
    dist = cv2.distanceTransform(src, cv2.DIST_L2, 5) / color.shape[1]
    return np.exp(-0.5 * (dist / SIGMA_D) ** 2)


def fraction(pop, field: np.ndarray) -> np.ndarray:
    """F on the left and the right. f_i is the mean of a over the cell's pixels."""
    height, width = field.shape
    ui = np.clip(np.rint(pop.u * width - 0.5).astype(np.int64), 0, width - 1)
    vi = np.clip(np.rint(pop.v * height - 0.5).astype(np.int64), 0, height - 1)
    ends = np.append(pop.starts[1:], len(pop.u))
    eff = np.empty(len(pop.starts))
    for i, (start, stop) in enumerate(zip(pop.starts, ends)):
        pixels = np.unique(vi[start:stop] * width + ui[start:stop])
        eff[i] = field[pixels // width, pixels % width].mean()
    out = np.zeros(2)
    for side in (0, 1):
        chosen = pop.side == side
        out[side] = float(eff[chosen] @ pop.w[chosen] / pop.side_sum[side])
    return out


def half_mask(left: bool, right: bool) -> np.ndarray:
    mask = np.zeros((HEIGHT, WIDTH), dtype=bool)
    mask[:, : WIDTH // 2] = left
    mask[:, WIDTH // 2 :] = right
    return mask


def rates(currents: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """One second of the poster membrane.

    Returns the mean, min, and max of the four firing rates over the last 100 ms.
    The filter still ripples between spikes, so a single sample is not the rate.
    """
    v = np.zeros(4)
    rate = np.zeros(4)
    silent_until = np.zeros(4)
    refrac_steps = int(round(REFRAC / DT))
    n = int(RUN / DT)
    tail = int(round(0.100 / DT))
    acc = np.zeros(4)
    lo = np.full(4, np.inf)
    hi = np.full(4, -np.inf)
    for step in range(n):
        spiked = np.zeros(4)
        for i in range(4):
            if step < silent_until[i]:
                v[i] = 0.0
                continue
            v[i] += (DT / TAU) * (-v[i] + currents[i])
            if v[i] >= 1.0:
                v[i] = 0.0
                silent_until[i] = step + 1 + refrac_steps
                spiked[i] = 1.0
        impulse = spiked / DT
        rate += (DT / TAU_R) * (-rate + impulse)
        if step >= n - tail:
            acc += rate
            lo = np.minimum(lo, rate)
            hi = np.maximum(hi, rate)
    return acc / tail, lo, hi


PACK = cells()
S = expansion(THETA_DOT)
print(f"alpha={ALPHA:.1f}  s(0)={expansion(0):.3f}  s(0.4)={S:.3f}  s(8)={expansion(8):.3f}")


alpha=2.0  s(0)=0.168  s(0.4)=0.179  s(8)=0.500


## 九通り

左半分と右半分を、青、赤、白で塗る。白は塗らない。青は LC4、赤は LPLC2 に入る。電流が 1 未満の膜はスパイクしない。

合否は `poster/color_wheel_table.md` の車輪と、計算した車輪が一致するかである。期待の列がその表である。


In [2]:
ROWS = [
    ("青", "青", True, True, False, False),
    ("青", "白", True, False, False, False),
    ("青", "赤", True, False, False, True),
    ("赤", "青", False, True, True, False),
    ("赤", "白", False, False, True, False),
    ("赤", "赤", False, False, True, True),
    ("白", "青", False, True, False, False),
    ("白", "白", False, False, False, False),
    ("白", "赤", False, False, False, True),
]

# poster/color_wheel_table.md の車輪。しみ出しは期待に入れていない。
WANT = {
    ("青", "青"): ("後", "後"),
    ("青", "白"): ("後", "強後"),
    ("青", "赤"): ("前", "後"),
    ("赤", "青"): ("後", "前"),
    ("赤", "白"): ("弱前", "前"),
    ("赤", "赤"): ("前", "前"),
    ("白", "青"): ("強後", "後"),
    ("白", "白"): ("止", "止"),
    ("白", "赤"): ("前", "弱前"),
}


def gear(speed: float) -> str:
    if speed > 1.0:
        return "前"
    if speed < -1.0:
        return "後"
    return "止"


print(
    f"{'L':2} {'R':2}  {'I DnaL':7} {'I DnaR':7} {'I DnpL':7} {'I DnpR':7}"
    f"  {'車輪':8} {'期待':10} 合否"
)
span = None
passed = 0
for left, right, blue_l, blue_r, red_l, red_r in ROWS:
    blue = spread(half_mask(blue_l, blue_r))
    red = spread(half_mask(red_l, red_r))
    f_lc4 = fraction(PACK.lc4, blue)
    f_lplc2 = fraction(PACK.lplc2, red)
    i_dna = ALPHA * f_lplc2
    i_dnp = ALPHA * S * f_lc4
    current = np.array([i_dna[0], i_dna[1], i_dnp[0], i_dnp[1]])
    fired, lo, hi = rates(current)
    if span is None and fired.max() > 1.0:
        span = (float(lo[fired > 1.0].min()), float(hi[fired > 1.0].max()))
    v_l = fired[1] - fired[3]
    v_r = fired[0] - fired[2]
    got = f"{gear(v_l)},{gear(v_r)}"
    want_l, want_r = WANT[(left, right)]
    want = f"{want_l},{want_r}"
    mark = "合" if got == want else "否"
    passed += mark == "合"
    print(
        f"{left:2} {right:2}  {i_dna[0]:7.3f} {i_dna[1]:7.3f} {i_dnp[0]:7.3f} {i_dnp[1]:7.3f}"
        f"  {got:8} {want:10} {mark}"
    )

print()
print(f"合 {passed}    否 {len(ROWS) - passed}")
if span is not None:
    print(f"発火した膜の発火率は、最後の 100 ms で {span[0]:.0f} から {span[1]:.0f} Hz を往復する。")
print("電流が 1 未満の膜はスパイクしない。k = 1 なので車輪の速さは Hz。")


L  R   I DnaL  I DnaR  I DnpL  I DnpR   車輪       期待         合否
青  青     0.000   0.000   0.359   0.359  止,止      後,後        否
青  白     0.000   0.000   0.359   0.118  止,止      後,強後       否
青  赤     0.461   2.000   0.359   0.118  前,止      前,後        否
赤  青     2.000   0.415   0.115   0.359  止,前      後,前        否
赤  白     2.000   0.415   0.000   0.000  止,前      弱前,前       否
赤  赤     2.000   2.000   0.000   0.000  前,前      前,前        合
白  青     0.000   0.000   0.115   0.359  止,止      強後,後       否
白  白     0.000   0.000   0.000   0.000  止,止      止,止        合
白  赤     0.461   2.000   0.000   0.000  前,止      前,弱前       否

合 2    否 7
発火した膜の発火率は、最後の 100 ms で 102 から 120 Hz を往復する。
電流が 1 未満の膜はスパイクしない。k = 1 なので車輪の速さは Hz。


## 読み

合は赤赤と白白の 2 行である。残り 7 行は否である。

DNp01 の電流は、画面いっぱいの青でも $\alpha s(0.4) = 2 \times 0.179 = 0.359$ で、閾値 1 に届かない。青を含む 5 行は後退が出ず、否である。

赤を塗った半面の DNa02 は電流 2 で、平均 111 Hz である。塗っていない側へ漏れた赤の電流は、左が 0.461、右が 0.415 で、どちらも閾値未満である。赤白の期待は弱前,前、白赤の期待は前,弱前である。計算は止,前と前,止なので、この 2 行も否である。鼻は赤のほうを向く。赤い側の車輪は弱く前へ出ずに止まる。

赤赤は両輪とも 111 Hz で前へ出て、期待と一致する。

ポスターの「赤い側の車輪は止まり、反対の車輪が前へ出る」だけを期待にすると、赤白と白赤は合になる。青の 5 行は、その読みでも否のままである。


## 半値を下げる

$m$ は $s$ が最大の半分になる拡大である。$k_s = 1$ のまま、$m$ を 0.4 rad/s より下に置くと、$s(0.4)$ は立ち上がりの上側に乗る。幅 $w$ が 5 rad/s のままだと、0 と 0.4 は同じ平らな場所にいる。幅も狭める。ここでは $m = 0.2$ rad/s、$w = 0.1$ rad/s とする。


In [3]:
def report_sigmoid(tag: str, mid: float, width: float) -> None:
    global M_S, W_S
    M_S, W_S = mid, width
    s0 = expansion(0.0)
    s4 = expansion(0.4)
    print(
        f"{tag}  m={mid:.2f}  w={width:.2f}"
        f"  s(0)={s0:.3f}  I={ALPHA * s0:.3f}"
        f"  s(0.4)={s4:.3f}  I={ALPHA * s4:.3f}"
    )


saved = (M_S, W_S, S)
report_sigmoid("幅が広いまま半値だけ下げる", 0.20, 5.0)
report_sigmoid("半値を刺激の 0.4 に置く", 0.40, 0.10)
report_sigmoid("半値を 0.2、幅を 0.1", 0.20, 0.10)

M_S, W_S = 0.20, 0.10
S = expansion(0.4)
print()
print(
    f"{'L':2} {'R':2}  {'I DnpL':7} {'I DnpR':7}"
    f"  {'r DnpL':7} {'r DnpR':7}  {'車輪':8} {'期待':10} 合否"
)
passed = 0
for left, right, blue_l, blue_r, red_l, red_r in ROWS:
    blue = spread(half_mask(blue_l, blue_r))
    red = spread(half_mask(red_l, red_r))
    f_lc4 = fraction(PACK.lc4, blue)
    f_lplc2 = fraction(PACK.lplc2, red)
    i_dna = ALPHA * f_lplc2
    i_dnp = ALPHA * S * f_lc4
    current = np.array([i_dna[0], i_dna[1], i_dnp[0], i_dnp[1]])
    fired, _, _ = rates(current)
    v_l = fired[1] - fired[3]
    v_r = fired[0] - fired[2]
    got = f"{gear(v_l)},{gear(v_r)}"
    want_l, want_r = WANT[(left, right)]
    want = f"{want_l},{want_r}"
    mark = "合" if got == want else "否"
    passed += mark == "合"
    print(
        f"{left:2} {right:2}  {i_dnp[0]:7.3f} {i_dnp[1]:7.3f}"
        f"  {fired[2]:7.1f} {fired[3]:7.1f}  {got:8} {want:10} {mark}"
    )

print()
print(f"合 {passed}    否 {len(ROWS) - passed}")
print("赤の電流と発火率は前の表と同じ。塗った青の半面だけが変わる。")

M_S, W_S, S = saved


幅が広いまま半値だけ下げる  m=0.20  w=5.00  s(0)=0.490  I=0.980  s(0.4)=0.510  I=1.020
半値を刺激の 0.4 に置く  m=0.40  w=0.10  s(0)=0.018  I=0.036  s(0.4)=0.500  I=1.000
半値を 0.2、幅を 0.1  m=0.20  w=0.10  s(0)=0.119  I=0.238  s(0.4)=0.881  I=1.762

L  R   I DnpL  I DnpR   r DnpL  r DnpR   車輪       期待         合否
青  青     1.762   1.762    111.1   111.1  後,後      後,後        合
青  白     1.762   0.577    111.1     0.0  止,後      後,強後       否
青  赤     1.762   0.577    111.1     0.0  前,後      前,後        合
赤  青     0.564   1.762      0.0   111.1  後,前      後,前        合
赤  白     0.000   0.000      0.0     0.0  止,前      弱前,前       否
赤  赤     0.000   0.000      0.0     0.0  前,前      前,前        合
白  青     0.564   1.762      0.0   111.1  後,止      強後,後       否
白  白     0.000   0.000      0.0     0.0  止,止      止,止        合
白  赤     0.000   0.000      0.0     0.0  前,止      前,弱前       否

合 5    否 4
赤の電流と発火率は前の表と同じ。塗った青の半面だけが変わる。


## 半値を下げた結果

$m = 0.2$ rad/s、$w = 0.1$ rad/s では、塗った青の半面が 111 Hz で発火する。合は 5 行になる。青青、青赤、赤青、赤赤、白白である。

残る否は、塗っていない側を弱く動かしたい 4 行である。その側の電流は、青で 0.56 前後、赤で 0.42 から 0.46 である。$s$ が 1 でも、この漏れは閾値 1 を超えない。
